In [1]:
# ============================================================
# ReAct Agent with Custom Tools
# ============================================================

# 1. Install dependencies
#!pip install -U langgraph langchain langchain-ollama langchain-core


# ============================================================
# 2. Imports
# ============================================================

from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

from langchain_ollama import ChatOllama
from langchain_core.tools import tool

from datetime import datetime
import ast
import operator


# ============================================================
# 3. Ollama Model
# ============================================================

model = ChatOllama(
    model="qwen3:0.6b",
    base_url="http://localhost:11434"
)


# ============================================================
# 4. Custom Tool 1 - Calculator
# ============================================================

@tool
def calculate(expression: str) -> str:
    """Safely calculate a mathematical expression."""

    try:
        allowed_operators = {
            ast.Add: operator.add,
            ast.Sub: operator.sub,
            ast.Mult: operator.mul,
            ast.Div: operator.truediv,
            ast.Pow: operator.pow,
            ast.Mod: operator.mod,
            ast.USub: operator.neg
        }

        def evaluate(node):

            if isinstance(node, ast.Constant):
                if isinstance(node.value, (int, float)):
                    return node.value
                raise ValueError("Invalid number")

            if isinstance(node, ast.BinOp):
                op = allowed_operators.get(type(node.op))

                if not op:
                    raise ValueError("Operator not allowed")

                return op(
                    evaluate(node.left),
                    evaluate(node.right)
                )

            if isinstance(node, ast.UnaryOp):
                op = allowed_operators.get(type(node.op))

                if not op:
                    raise ValueError("Operator not allowed")

                return op(evaluate(node.operand))

            raise ValueError("Invalid expression")

        tree = ast.parse(
            expression,
            mode="eval"
        )

        result = evaluate(tree.body)

        return str(result)

    except Exception as e:
        return f"Error: {e}"


# ============================================================
# 5. Custom Tool 2 - Dictionary
# ============================================================

@tool
def define_word(word: str) -> str:
    """Return the definition of a word."""

    definitions = {
        "artificial": "Made or produced by humans rather than occurring naturally.",
        "intelligence": "The ability to learn, understand, reason, and solve problems.",
        "photosynthesis": "The process by which plants use sunlight to make food from carbon dioxide and water.",
        "algorithm": "A step-by-step procedure for solving a problem or completing a task.",
        "machine": "A device or system designed to perform a particular task."
    }

    word = word.lower().strip()

    return definitions.get(
        word,
        f"No definition found for '{word}'."
    )


# ============================================================
# 6. Custom Tool 3 - Current Date/Time
# ============================================================

@tool
def get_current_datetime() -> str:
    """Return the current date and time."""

    return datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )


# ============================================================
# 7. Create ReAct Agent
# ============================================================

tools = [
    calculate,
    define_word,
    get_current_datetime
]

memory = MemorySaver()

agent = create_react_agent(
    model,
    tools,
    checkpointer=memory
)


# ============================================================
# 8. Helper Function
# ============================================================

def ask_agent(question, thread_id="session1"):

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        }
    )

    return response




/Users/tech/PycharmProjects/PythonProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/4t/g31q231s31s2mvmbd_4pv__h0000gr/T/ipykernel_36241/949628578.py:143: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [3]:
# ============================================================
# 9. Test 1 - Calculator Tool
# ============================================================

response = ask_agent(
    "What is 125 * 24 + 50?"
)

print(response["messages"][-1].content)


The result of $125 \times 24 + 50$ is 3050. Let me know if you need further assistance!


In [4]:
# ============================================================
# 10. Test 2 - Dictionary Tool
# ============================================================

response = ask_agent(
    "What does photosynthesis mean?"
)

print(response["messages"][-1].content)

Photosynthesis is the process by which plants use sunlight, carbon dioxide, and water to produce glucose and oxygen. It is the fundamental process by which plants generate energy for their own survival and the food chain they support.


In [5]:
# ============================================================
# 11. Test 3 - Date/Time Tool
# ============================================================

response = ask_agent(
    "What is the current date and time?"
)

print(response["messages"][-1].content)

The current date and time is 2026-08-19 19:02:54.


In [6]:
# ============================================================
# 12. Test 4 - Direct Question
# ============================================================

response = ask_agent(
    "What is the capital of France?"
)

print(response["messages"][-1].content)


The capital of France is Paris.


In [7]:
# ============================================================
# 13. Test 5 - Direct Question
# ============================================================

response = ask_agent(
    "Why is the sky blue?"
)

print(response["messages"][-1].content)

The sky appears blue because light is scattered by the Earth's atmosphere. Specifically, the blue light is scattered more by the atmosphere than other colors, creating the blue hue. This effect is called Rayleigh scattering.


In [8]:
# ============================================================
# 14. Test 6 - Tool + Reasoning
# ============================================================

response = ask_agent(
    "Calculate 250 * 12. If the result is greater than 2000, "
    "tell me that the result is above the threshold."
)

print(response["messages"][-1].content)

The result of 250 * 12 is 3000, which is greater than 2000. The result is above the threshold.


In [16]:
# ============================================================
# 15. Conversation Memory Test
# ============================================================

thread_id = "memory-demo"

response = ask_agent(
    "My name is Vivek.",
    thread_id
)

response = ask_agent(
    "What is my name?",
    thread_id
)

print(response["messages"][-1].content)


Hello, Vivek! What can I assist you with right now?


In [17]:
# ============================================================
# 16. Show ReAct Loop
# ============================================================

response = ask_agent(
    "Calculate 45 * 12 and tell me whether the result is greater than 500."
)

for message in response["messages"]:

    print("\n" + "=" * 60)
    print("MESSAGE TYPE:", message.type)
    print("=" * 60)

    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("\nTOOL CALL:")
        print(message.tool_calls)


MESSAGE TYPE: human
What is 125 * 24 + 50?

MESSAGE TYPE: ai


TOOL CALL:
[{'name': 'calculate', 'args': {'expression': '125 * 24 + 50'}, 'id': '5af71ce8-45e4-4bca-8e72-afeb9cbe758a', 'type': 'tool_call'}]

MESSAGE TYPE: tool
3050

MESSAGE TYPE: ai
The result of $125 \times 24 + 50$ is 3050.

MESSAGE TYPE: human
What is 125 * 24 + 50?

MESSAGE TYPE: ai
The result of $125 \times 24 + 50$ is 3050. Let me know if you need further assistance!

MESSAGE TYPE: human
What does photosynthesis mean?

MESSAGE TYPE: ai


TOOL CALL:
[{'name': 'define_word', 'args': {'word': 'photosynthesis'}, 'id': '039754af-5f8b-4cf1-be05-70cf5ceec0bb', 'type': 'tool_call'}]

MESSAGE TYPE: tool
The process by which plants use sunlight to make food from carbon dioxide and water.

MESSAGE TYPE: ai
Photosynthesis is the process by which plants use sunlight, carbon dioxide, and water to produce glucose and oxygen. It is the fundamental process by which plants generate energy for their own survival and the food chain 